# Lesson 02 Lab — Prefill, Decode, and the KV Cache

**Puzzle:** Which phase owns TTFT, which phase owns ITL, and why does context remain resident?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A long prompt and a long answer stress different parts of the engine. Collapsing them into one requests-per-second number hides whether attention over the prompt, repeated Decode steps, or KV capacity is the limiting factor.


## 0. Predict before running

1. Rank the four workload cells by expected elapsed time.
2. Calculate which cells reserve the most KV positions.
3. State whether offline RequestOutput metrics expose true network-observed TTFT.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The experiment creates short- and long-prompt requests with short and long output limits, runs them through one native engine, and retains request metrics when exposed by the installed API. Token counts provide the fallback ledger.

- Prompt length primarily changes the initial compute and cache allocation.
- Output length repeats Decode and grows the cache one token at a time.
- Aggregate elapsed time cannot identify TTFT without first-token timing.


## 2. Derive the mechanism

Prefill maps all prompt tokens through the model and materializes key/value vectors for every layer. Decode reuses that state and appends one position per step. For a standard decoder, KV bytes scale approximately with `2 × layers × tokens × kv_heads × head_dim × bytes_per_element`. TTFT contains queue plus prompt work; ITL reflects the sequence of Decode scheduling and execution events.

### Mechanism at a glance

```mermaid
flowchart LR
  P["prompt tokens"] --> F["Prefill: many positions"]
  F --> K["layer KV cache"]
  K --> D["Decode: one new position"]
  D --> K
  D --> T["next token"]
  F -.-> A["TTFT path"]
  D -.-> I["ITL path"]
```

### Walk it step by step

1. **Tokenize first.** The prompt-token count defines initial work and cache positions.
2. **Materialize reusable state.** Prefill writes keys and values for every layer.
3. **Append during Decode.** Each generated token extends that state and triggers another model step.
4. **Attach the right metric.** Use TTFT for initial work and ITL for repeated generation.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 2
LESSON_TITLE = 'Prefill, Decode, and the KV Cache'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260814
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | short prompt with an eight-token answer |
| Candidate | long prompt and/or a 32-token answer |
| Held constant | engine, model, dtype, seed, sampling mode, and GPU |
| Measurements | prompt tokens, output tokens, elapsed time, and available request metrics |
| Evidence | `native-backend` |

**Experiment:** Measure a 2×2 prompt/output grid through vLLM and retain token and request timing fields.


## 5. Inspect the experiment code

Each workload runs as a separate native request after one warm-up. The code introspects the metrics object instead of assuming version-specific attributes, so absent fields remain explicit rather than fabricated.

Do not execute until the code matches the frozen table.


In [2]:
llm = LLM(**base_engine_args(max_model_len=2048)); short = "Explain TTFT versus ITL."
long = ("Prefill processes prompts and Decode reuses cached key/value vectors. " * 70) + "Summarize."
cases = {"short_short": (short, 8), "long_short": (long, 8),
         "short_long": (short, 32), "long_long": (long, 32)}; rows = {}
for name, (prompt, limit) in cases.items():
    tick = time.perf_counter()
    item = llm.generate([prompt], SamplingParams(temperature=0.0, max_tokens=limit, seed=SEED),
                        use_tqdm=False)[0]
    row = output_record(item); row["elapsed_s"] = time.perf_counter() - tick
    state = getattr(item, "metrics", None)
    row["request_metrics"] = ({key: getattr(state, key, None) for key in
        ("arrival_time", "first_token_time", "finished_time", "scheduler_time",
         "model_forward_time", "model_execute_time")} if state else {})
    rows[name] = row
metrics = {"cases": rows}
analysis = (f"The long prompt used {rows['long_short']['prompt_tokens']} tokens versus "
            f"{rows['short_short']['prompt_tokens']} short; the long answer produced "
            f"{rows['short_long']['output_tokens']} tokens. Elapsed time combines phases, and only "
            "non-null native request fields count as phase timing evidence.")


INFO 08-13 00:15:33 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260814, 'max_model_len': 2048, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:15:33 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:15:33 [model.py:1883] Using max model len 2048


INFO 08-13 00:15:33 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:15:33 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=649466) 2026-08-13 00:15:45,540 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=649466) 2026-08-13 00:15:45,598 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=649466) 2026-08-13 00:15:45,630 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/a10ba88b4090547465f2afcd30e11488df4381edf2bc37e4f4901ce2732cfd44/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=649466) INFO 08-13 00:15:46 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=649466) INFO 08-13 00:15:46 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.73 s


(EngineCore pid=649466) WARNING 08-13 00:15:46 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=649466) WARNING 08-13 00:15:46 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=649466) INFO 08-13 00:15:46 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=649466) INFO 08-13 00:15:46 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=649466) INFO 08-13 00:15:47 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:15:47 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Short/short elapsed | 0.084925 |
| Long/short elapsed | 54.500730 |
| Short/long elapsed | 0.255930 |
| Long/long elapsed | 0.268379 |
| Longest prompt tokens | 914 |
| Longest output tokens | 32 |


## 7. Explain the result

The long prompt used 914 tokens versus 8 short; the long answer produced 32 tokens. Elapsed time combines phases, and only non-null native request fields count as phase timing evidence.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 2, "title": 'Prefill, Decode, and the KV Cache', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Prefill, Decode, and KV growth are distinct mechanisms; this run measures their combined native request cost and exposes only the timing fields the API actually returns.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 2,
  "title": "Prefill, Decode, and the KV Cache",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260814
  },
  "evidence_label": "native-backend",
  "metrics": {
    "cases": {
      "short_short": {
        "request_id": "0",
        "prompt_tokens": 8,
        "output_tokens": 8,
        "token_ids": [
          3555,
          374,
          279,
          6672,
          1948,
          1493,
          1378,
          4494
        ],
        "text_preview": " What is the difference between these two types",
        "text_sha256": "4e7421f214d9b1bf2b79f8fcebe773a7e87c7d5844f66a37f0f01bbb4dba62a6",
        "finish_reason": "length",
        "stop_reason": null,
        "num_cached_tokens": 0,
        "elapsed_s": 0.08492478681728244,
        "request_metrics": {}
     

## 9. Make the bounded decision

> Prefill, Decode, and KV growth are distinct mechanisms; this run measures their combined native request cost and exposes only the timing fields the API actually returns.

**Acceptance/rollback:** Use phase-specific optimization only after the metric that corresponds to that phase moves under a representative workload.

**Failure analysis:** Wall-clock measurements include Python and scheduler overhead. Repeated prose may tokenize differently than expected, and an offline first-token timestamp is not client-observed streaming latency.


## 10. Extend the evidence

Run the same grid over the streaming API, timestamp every chunk at the client, and compare engine timestamps with network observations.

The full boundary and references are in [`README.md`](README.md).
